# Assignment 4 — Part 2: common evidence notebook

Ordinary single-table Q-learning versus PPO in native Gymnasium MsPacMan.
This notebook combines the reviewed workflow with a shared results/inspection interface.
It never trains automatically and never chooses models from test performance.

The original Part 1 predictions and all registered runtime source bytes are preserved.
The report uses one controlled A–D protocol, rather than pooling incompatible branch results.
See `synthesis/COMPARISON.md` for the four-branch decisions and exact commits.


In [ ]:
from pathlib import Path
import sys
import json

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'IT3212 - Assignment 4.pdf').exists())
sys.path.insert(0, str(ROOT / 'notebooks/assignment4/synthesis'))
from part2_evidence import load_bundle, check_bundle, dashboard
from IPython.display import display, Markdown, Image, HTML
import pandas as pd

bundle = load_bundle()
# Missing ignored records are reported explicitly on a fresh checkout.
# Use full=True when the original local results archive is present.
print(json.dumps(check_bundle(full=False), indent=2))


## Task and comparison contract

- A: maze 0, sticky probability 0. B: maze 0, probability 0.25. C: B minus 0.1 points per decision.
- A/B use instructor-approved native score-increment rewards; both methods learn from reward / 100.
- Success: 2,000 newly earned points before game over or 1,000 decisions. This is not maze completion.
- D: all frozen B models on maze 1, with compatible observations/actions and zero additional training.
  Native map exits are censored separately from failures; crossing rewards are excluded from prefix returns.
- Common cap: 262,144 decisions. Five common training seeds: 17, 29, 101, 314, 2718.
  Selected checkpoints have 131,072–262,144 transitions; the full consumed cap remains charged.
- Equal new search: seven packages per method on A/B at seed 42, top two repeated at seed 7.
  Earlier cumulative development was unequal and is disclosed. This is not Part 3's mechanism study.
- Final evaluation: 200 episodes per model/condition, no updates. Greedy Q with seeded ties; PPO argmax.
  Main A/B/C evaluation seeds: 1150000–1150199. Repaired D seeds: 1250000–1250199.

Seeds are phase identifiers, not values selected for performance. Five training seeds are the replications;
1,000 evaluation episodes per condition are not 1,000 independently trained agents.


In [ ]:
summary = pd.DataFrame(bundle['summaries'])
columns = ['condition', 'algorithm', 'success_rate', 'success_rate_sd',
           'failure_rate', 'censored_rate', 'mean_return', 'mean_return_sd',
           'success_length', 'failure_length']
display(summary[columns].round(4))


## All training replications

A PPO average of 80% contains four 100% agents and one 0% agent.
D rates describe observed in-map success. Q has one censored episode among 1,000,
so its aggregate logical success bound is 4.5–4.6%, not a confidence interval.
Compare B/C using success or raw score; C's penalised return has a different reward definition.


In [ ]:
agents = pd.DataFrame(bundle['agents'])
per_seed = 100 * agents.pivot(index=['training_seed', 'algorithm'],
                             columns='condition', values='success_rate')
display(per_seed)
display(agents[['condition', 'algorithm', 'training_seed', 'training_interactions',
                'checkpoint_interactions', 'training_seconds', 'evaluation_seconds',
                'episodes', 'censored']].round(2))


## Learning against interactions

Source: 22 November 2026 main experiment, seeds 17/29/101/314/2718.
Unsmoothed development success checked every 65,536 interactions, ten episodes per model.
Shading is sample SD across trained agents, not a confidence interval. Final tests are excluded.


In [ ]:
display(Image(filename=str(ROOT / 'reports/assignment4/Images/part2_learning_curves_20261122.png')))


## Shared overview and native episode inspection

Create a self-contained overview (no server or network required):


In [ ]:
overview = ROOT / 'results/assignment4/synthesis/overview.html'
dashboard(bundle, overview)
print(overview)


Inspect an explicitly selected recorded episode, with native Gymnasium replay:

```powershell
python notebooks/assignment4/synthesis/replay.py --condition B --algorithm PPO --training-seed 17 --evaluation-seed 1150000 --output results/assignment4/synthesis/B_PPO17_1150000.html
```

The viewer displays the pre-action native screen, chosen action, recorded Q-values or policy probabilities,
score/reward transition and observation. Every intervening action is replayed even if the display skips frames.
The tool refuses incompatible source/model/trace hashes and does not automatically pick a high-scoring test model.
A replay is a verification of existing evidence, not a new test or a counterfactual experiment.
The original local records must be available; the compact snapshot alone cannot reconstruct RGB.

## Reproduce or audit

Read-only export and full original-archive check:

```powershell
python notebooks/assignment4/synthesis/part2_evidence.py build
python notebooks/assignment4/synthesis/part2_evidence.py check --full
python notebooks/assignment4/report_review/analyse.py
python notebooks/assignment4/report_review/verify_quality.py
```

Training is deliberately explicit and expensive. The exact original search/training sequence is documented
in `three_hour/README.md`, and the boundary repair in `transfer_evaluation/README.md`.
Use a new run ID and preserve the historical sources/results. Reproduction on the original seeds
is not a new independent validation. Do not use historical renderers to overwrite the reviewed report.

This notebook covers Part 2. Parts 3–6 require their separate work; no human review is invented here.
